# v76 -- Spiral Correlation: KTF³-R Smoking Gun
## Peaks at k×1660 Mpc + k×25.714° rotation
**Author:** Andrew Cotting | 5 April 2026 | github.com/Andrew-Cot/KTF3-Analysis

### Prediction
Z14 screw space: galaxy at x has correlated partner at φ_R^k(x).
2-point correlation should show peaks at k×T₁ = k×1660 Mpc.
This notebook uses SDSS DR12 spectroscopic galaxies via astroquery.

In [ ]:
!pip install astroquery astropy -q
import numpy as np, matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from astropy.cosmology import FlatLambdaCDM
import os

print('='*60)
print('v76 -- Spiral Correlation: KTF³-R Smoking Gun')
print('T₁=1660 Mpc, θ_R=25.714° | SDSS DR12 galaxies')
print('='*60)

T1 = 1660.0
theta_R = np.radians(25.714)
cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)

l_k, b_k = np.radians(277.0), np.radians(-31.0)
axis_vec = np.array([np.cos(b_k)*np.cos(l_k),
                     np.cos(b_k)*np.sin(l_k),
                     np.sin(b_k)])
print(f'KTF³ axis: {axis_vec.round(4)}')
print(f'T₁ = {T1} Mpc, θ_R = {np.degrees(theta_R):.3f}°')


In [ ]:
from astroquery.sdss import SDSS

sdss_file = 'sdss_dr12_galaxies.npy'
if os.path.exists(sdss_file):
    print('Loading cached SDSS data...')
    data = np.load(sdss_file)
    ra, dec, z_spec = data[:,0], data[:,1], data[:,2]
    is_real = True
else:
    print('Querying SDSS DR12 (1-2 min)...')
    query = """
    SELECT TOP 50000 p.ra, p.dec, s.z
    FROM PhotoObj p JOIN SpecObj s ON p.objID = s.bestObjID
    WHERE s.class = 'GALAXY' AND s.z BETWEEN 0.05 AND 0.55
    AND s.zWarning = 0 AND p.ra BETWEEN 100 AND 260
    AND p.dec BETWEEN -10 AND 60 ORDER BY NEWID()
    """
    try:
        result = SDSS.query_sql(query, timeout=120)
        ra = result['ra'].data.astype(float)
        dec = result['dec'].data.astype(float)
        z_spec = result['z'].data.astype(float)
        np.save(sdss_file, np.column_stack([ra, dec, z_spec]))
        is_real = True
        print(f'Downloaded {len(ra):,} galaxies')
    except Exception as e:
        print(f'Query failed: {e} -- synthetic')
        is_real = False
        np.random.seed(277)
        n = 20000
        ra = np.random.uniform(100, 260, n)
        dec = np.random.uniform(-10, 60, n)
        z_spec = np.random.uniform(0.05, 0.55, n)

d_com = cosmo.comoving_distance(z_spec).value
ra_r = np.radians(ra); dec_r = np.radians(dec)
pos = np.column_stack([
    d_com*np.cos(dec_r)*np.cos(ra_r),
    d_com*np.cos(dec_r)*np.sin(ra_r),
    d_com*np.sin(dec_r)
])
print(f'N={len(pos):,}, d={d_com.min():.0f}-{d_com.max():.0f} Mpc')


In [ ]:
# 2-point correlation
n_sample = min(3000, len(pos))
idx = np.random.choice(len(pos), n_sample, replace=False)
pos_s = pos[idx]

print('Computing pairwise distances...')
dists = cdist(pos_s, pos_s)
dists_flat = dists[dists > 0].flatten()

r_max = min(d_com.max(), 5000)
r_bins = np.linspace(0, r_max, 150)
r_centers = 0.5*(r_bins[1:]+r_bins[:-1])
hist_obs, _ = np.histogram(dists_flat, bins=r_bins)

# Random catalog
ra_rand = np.random.uniform(ra.min(), ra.max(), n_sample)
dec_rand = np.random.uniform(dec.min(), dec.max(), n_sample)
d_rand = np.random.choice(d_com, n_sample)
ra_r2 = np.radians(ra_rand); dec_r2 = np.radians(dec_rand)
pos_rand = np.column_stack([
    d_rand*np.cos(dec_r2)*np.cos(ra_r2),
    d_rand*np.cos(dec_r2)*np.sin(ra_r2),
    d_rand*np.sin(dec_r2)
])
dists_rand = cdist(pos_rand, pos_rand)
hist_rand, _ = np.histogram(dists_rand[dists_rand>0].flatten(), bins=r_bins)

hist_rand_safe = np.where(hist_rand > 0, hist_rand, 1)
xi = hist_obs / hist_rand_safe - 1
xi_smooth = np.convolve(xi, np.ones(5)/5, mode='same')

print('KTF³ peak check:')
for k in range(1, 5):
    d_k = k * T1
    if d_k < r_max:
        idx_k = np.argmin(np.abs(r_centers - d_k))
        print(f'  k={k}: {d_k:.0f} Mpc → ξ={xi_smooth[idx_k]:+.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16,6))
fig.suptitle(f'v76 -- Spiral Correlation KTF³-R | {"SDSS DR12" if is_real else "SYNTHETIC"}', fontsize=13, fontweight='bold')

ax1 = axes[0]
ax1.plot(r_centers, xi_smooth, 'b-', lw=1.5)
ax1.axhline(0, color='black')
for k in range(1,4):
    d_k = k*T1
    if d_k < r_max:
        ax1.axvline(d_k, color='red', linestyle='--', lw=1.5, label=f'k={k}: {d_k:.0f}Mpc')
ax1.set_xlabel('r [Mpc]'); ax1.set_ylabel('ξ(r)')
ax1.set_title('2-point correlation + KTF³ scale predictions')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = axes[1]
mask = (r_centers > 1000) & (r_centers < 3500)
ax2.plot(r_centers[mask], xi_smooth[mask], 'g-o', lw=2, ms=4)
ax2.axhline(0, color='black')
for k in [1,2]:
    d_k = k*T1
    if 1000 < d_k < 3500:
        ax2.axvline(d_k, color='red', linestyle='--', lw=2, label=f'k={k}: {d_k:.0f}Mpc')
ax2.set_xlabel('r [Mpc]'); ax2.set_ylabel('ξ(r)')
ax2.set_title('Zoom: KTF³ range | Definitive: Euclid DR1 Oct 2026')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v76_spiral_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('GitHub: github.com/Andrew-Cot/KTF3-Analysis | OSF: osf.io/42nks')
